# Протоколы взаимодействия AI-агентов

**Тращенков Сергей · МФТИ · «Разработка и внедрение AI-агентов»**

Блокнот для занятия на 90 минут и продолжения на 45 минут. Дополнительный раздел посвящён Skills.
Мы собираем помощника для поездки: он ищет билеты на Туту, подбирает продукты во ВкусВилле,
обращается к другому агенту и передаёт результаты пользовательскому приложению.

**Первое занятие:** MCP, готовые серверы, tools/resources/prompts, собственный сервер, изменения спецификации и LangChain.
**Продолжение:** два способа публикации A2A, координатор, AG-UI, A2UI, Agent Client Protocol, обзор коммерческих протоколов.

Установка одна на весь репозиторий и описана в [README](README.md). Перед работой заполните `.env`.
Ячейки выполняют реальные запросы: ответы модели и результаты внешнего поиска могут меняться.
Запускайте модельные примеры последовательно. Для повторного чтения достаточно сохранённого вывода ячейки.

## 1. Участники взаимодействия и общая настройка

MCP связывает приложение с сервером инструментов и контекста. A2A связывает приложение с удалённым агентом.
AG-UI описывает события между агентом и пользовательским приложением. A2UI описывает создаваемый агентом интерфейс.
Agent Client Protocol связывает редактор кода и агентный процесс. Навыки (Skills) содержат инструкции и вспомогательные файлы.

В MCP **host** — приложение, в котором работает агент. Оно создаёт MCP-клиента и подключает серверы.
Модель получает описания инструментов от приложения. При выборе инструмента приложение передаёт запрос серверу,
а полученный результат добавляет в историю диалога. Вычисления самого инструмента выполняет сервер.

Здесь используется Python из окружения `uv`. В Jupyter доступны ячейки с `await`; отдельный `asyncio.run()` в них не нужен.
Следующая ячейка загружает настройки, задаёт будущую дату поездки и импортирует служебные средства запуска процессов.
Она нужна также при начале работы сразу со вторым занятием после перезапуска ядра.

In [ ]:
import asyncio
import json
import os
import sys
from pathlib import Path
from collections import Counter

import httpx
from IPython.display import display, HTML, Markdown
from langchain.mcp import MCPAdapter
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from deepagents import create_deep_agent

import lab
from lab import ROOT, make_model

TRIP_DATE = lab.future_date(21)
QUESTION = f"Найди самый дешёвый прямой поезд Москва — Казань на {TRIP_DATE}. Укажи номер, отправление, прибытие и цену. Только поиск."
print("Python:", sys.version.split()[0])
print("Дата поездки:", TRIP_DATE)
print("Модель:", os.getenv("GIGACHAT_MODEL", "GigaChat-3-Ultra"))
print("Ключ задан:", bool(os.getenv("GIGACHAT_CREDENTIALS")))
assert os.getenv("GIGACHAT_CREDENTIALS"), "Заполните GIGACHAT_CREDENTIALS в .env и повторите ячейку"

## 2. Три основных примитива MCP

У MCP-сервера есть три способа предоставить приложению возможности:

| Примитив | Пример | Как используется |
|---|---|---|
| **Tool** | Поиск билетов по городам и дате | Модель выбирает вызов, приложение исполняет его через MCP |
| **Resource** | Описание курса по URI `course://overview` | Приложение читает данные и решает, что передать модели |
| **Prompt** | Шаблон справки с параметрами темы и аудитории | Пользователь через приложение выбирает шаблон; сервер формирует сообщения |

Получение списка инструментов ещё не запускает модель. Ресурсы и prompts сами по себе также не появляются в её контексте:
приложение должно получить их и собрать входные сообщения. В разделе 6 мы увидим весь этот путь на собственном сервере.
В протоколе также есть обнаружение возможностей, сообщения об ошибках, уведомления и запросы дополнительного ввода.

## 3. Изменения спецификации, SDK и LangChain

Версию спецификации MCP обозначают датой. Версия Python SDK имеет собственный номер.
В этом занятии сравниваются редакции спецификации **2025-11-25** и **2026-07-28**, а также Python SDK v1 и v2.

| Уровень | Прежний вариант | Используемый вариант |
|---|---|---|
| Начало обмена | `initialize`, согласование возможностей, сессия HTTP | Версия и возможности в `_meta` каждого запроса, `server/discover` |
| Межвызовное состояние | Идентификатор сессии транспорта | Явные идентификаторы объектов в аргументах инструментов |
| Python SDK | `FastMCP` | `MCPServer`, новый `Client`, настройка транспорта в `run()` |
| LangChain | Пакет `langchain-mcp-adapters`, `MultiServerMCPClient` | `langchain[mcp]`, `MCPAdapter`, async context manager и `list_tools()` |

Изменения затронули и дополнительные механизмы. Запрос дополнительного ввода представлен многошаговым обменом MRTR:
сервер возвращает `input_required`, приложение получает ввод и повторяет запрос. Подписки собраны в `subscriptions/listen`.
Tasks вынесены в расширение. Roots, Sampling и протокольное Logging помечены устаревающими.
Streamable HTTP может передавать SSE-поток; устаревающим объявлен отдельный старый транспорт HTTP+SSE.

`langchain.mcp` находится в beta. Он оборачивает tools; resources и prompts ниже читаем через Python SDK MCP.
Источники: [изменения MCP](https://modelcontextprotocol.io/specification/2026-07-28/changelog),
[SDK v2](https://py.sdk.modelcontextprotocol.io/whats-new/),
[миграция LangChain](https://docs.langchain.com/oss/python/migrate/langchain-mcp-adapters).

## 4. Туту: сначала обнаруживаем возможности

Подключаем публичный сервер Туту по HTTP. `MCPAdapter` — клиентская обёртка в нашем приложении;
она преобразует описания серверных tools в инструменты LangChain. Контекст подключения держим открытым на время работы.

В следующей ячейке модель не используется: получаем только каталог. Число и состав инструментов у внешнего сервиса
могут измениться. Посмотрите, какие задачи обещают решить описания и какие параметры потребуются для их выполнения.

In [ ]:
async with MCPAdapter("https://mcp.tutu.ru/mcp") as adapter:
    tutu_tools = await adapter.list_tools()
    for t in tutu_tools:
        print(t.name, "—", t.description.strip().splitlines()[0][:130])
    rail_tool = next(t for t in tutu_tools if t.name.endswith("search_rail"))
    rail_schema = rail_tool.args_schema
print("Инструментов:", len(tutu_tools))

### 4.1. Схема инструмента и необязательные параметры

JSON Schema задаёт структуру аргументов. Поле `required` сообщает, какие свойства обязательны.
`anyOf` разрешает несколько вариантов значения: например, целое число либо `null`.
Схема поступает от внешнего MCP-сервера, поэтому совместимость нужно проверять по всей цепочке:
MCP → адаптер LangChain → клиент модели → API модели.

В выбранных версиях GigaChat API v2 работает с такими схемами; параметр `use_api_v2=True` выставляется в `make_model()`.
Команда `uv sync` устанавливает версии из `uv.lock`.
Далее используем исходные схемы, которые отдаёт сервер.

In [ ]:
schema = rail_schema if isinstance(rail_schema, dict) else rail_schema.model_json_schema()
print(json.dumps(schema, ensure_ascii=False, indent=2)[:9000])

### 4.2. Где собирается и запускается агент

Список `tools` передаём в `create_deep_agent`. В `ainvoke` передаём сообщения пользователя.
Агент может сделать несколько шагов: вызвать инструмент, прочитать результат, уточнить поиск и сформировать ответ.
В этой ячейке явно видны и сборка агента, и его запуск.

`recursion_limit` ограничивает число шагов графа. Это техническая граница выполнения, достижение которой не означает,
что агент успешно ответил. В учебном запуске после найденного результата просим завершить ответ.

In [ ]:
model = make_model()
try:
    async with MCPAdapter("https://mcp.tutu.ru/mcp") as adapter:
        tools = await adapter.list_tools()
        agent = create_deep_agent(
            model=model, tools=tools,
            system_prompt="Ищи сведения о поездке инструментами Туту. После успешного поиска дай краткий ответ и заверши работу. Не оформляй заказ.",
        )
        tutu_result = await agent.ainvoke(
            {"messages": [HumanMessage(content=QUESTION)]}, config={"recursion_limit": 60},
        )
finally:
    await model._client.aclose()
for message in tutu_result["messages"]:
    message.pretty_print()

### 4.3. Как читать историю

Найдите в `AIMessage` вызов `search_rail` и сравните его аргументы с исходным запросом.
В `ToolMessage` находится ответ сервиса. Следующий `AIMessage` использует полученные данные.
Проверьте города, дату, сортировку и наличие условия прямого маршрута. Цена и доступность относятся к моменту поиска.

**Вопрос:** что изменится, если пользователь не укажет дату? В каких случаях агент должен уточнить условие?

## 5. Собственный MCP-сервер

Сервер `servers/course_mcp.py` публикует учебные файлы из `data/course`. В нём есть два ресурса, prompt и два инструмента.
Откройте определения и найдите декораторы `@server.resource`, `@server.prompt` и `@server.tool`.
В `read_artifact_head` путь разрешается внутри папки данных; попытка выйти за её пределы отклоняется.

Сначала используем транспорт stdio: MCP-клиент запускает Python-сервер как отдельный дочерний процесс.
Данные протокола идут по stdin/stdout. В Python-клиенте это обычный async context manager;
он отвечает за соединение и завершение дочернего процесса.

In [ ]:
display(Markdown("```python\n" + (ROOT / "servers/course_mcp.py").read_text(encoding="utf-8") + "\n```"))

### 5.1. Три списка: tools, resources, prompts

Получим все три каталога через сырой клиент SDK. Это даёт полную картину возможностей сервера.
После выхода из `async with` сервер stdio будет остановлен. Описания и прочитанные данные останутся в переменных ноутбука.

In [ ]:
from mcp import Client, StdioServerParameters

params = StdioServerParameters(command=sys.executable, args=[str(ROOT / "servers/course_mcp.py")])
async with Client(params) as client:
    listed_tools = await client.list_tools()
    listed_resources = await client.list_resources()
    listed_prompts = await client.list_prompts()
    print("TOOLS:", [t.name for t in listed_tools.tools])
    print("RESOURCES:", [str(r.uri) for r in listed_resources.resources])
    print("PROMPTS:", [p.name for p in listed_prompts.prompts])
    resource = await client.read_resource("course://overview")
    prepared = await client.get_prompt("prepare_lecture_brief", {"topic_name": "mcp", "audience": "студенты МФТИ"})
    overview_text = resource.contents[0].text
    prompt_texts = [m.content.text for m in prepared.messages]
print("RESOURCE:\n", overview_text)
print("PROMPT:\n", "\n".join(prompt_texts))

## 6. Собираем вход модели: resource + prompt + tools

Вход формирует наше приложение. Оно прочитало ресурс, запросило шаблон с аргументами и теперь создаёт сообщения модели.
Покажем весь получившийся контекст до запуска. Сам по себе URI ресурса не заставляет модель прочитать его.

Подключим tools через `MCPAdapter`. Во время ответа агент сможет запросить список материалов темы и прочитать начало файла.
Так в одном сценарии участвуют все три примитива: resource предоставляет исходный контекст, prompt задаёт структуру задачи,
а tools дают возможность получить подробности во время выполнения.

In [ ]:
brief_messages = [HumanMessage(content=t) for t in prompt_texts]
brief_messages.append(HumanMessage(content=f"Контекст курса из MCP-ресурса:\n{overview_text}\n\nПолучите список файлов темы mcp и прочитайте первые 6 строк первого файла. Затем подготовьте справку."))
for message in brief_messages:
    message.pretty_print()

In [ ]:
model = make_model()
try:
    async with MCPAdapter(ROOT / "servers/course_mcp.py") as adapter:
        course_agent = create_deep_agent(model=model, tools=await adapter.list_tools())
        brief = await course_agent.ainvoke({"messages": brief_messages}, config={"recursion_limit": 60})
finally:
    await model._client.aclose()
for message in brief["messages"]:
    message.pretty_print()

## 7. Тот же сервер по HTTP и сравнение редакций MCP

Включим у того же Python-сервера Streamable HTTP. Ячейка запуска возвращает управление после проверки готовности.
Объект процесса нужен для чтения журнала и остановки. Повторный запуск одноимённого сервера сначала остановит старый
процесс, запущенный этим блокнотом. Занятые посторонними процессами порты автоматически не освобождаются.

In [ ]:
mcp_http = lab.start_server("course-mcp", "servers/course_mcp.py", port=8765,
                            ready_path="/mcp", args=["--http", "--port", "8765"])

### 7.1. Запрос без предварительного initialize

Сначала получим сведения о сервере через `server/discover`, затем вызовем инструмент.
В запросах передаются версия протокола и возможности клиента. Посмотрите `_meta`, имя метода и ответ `resultType`.
В примере `tools/call` предварительного рукопожатия нет.

In [ ]:
request, discovery, headers = await lab.mcp_request("server/discover")
print("REQUEST:", json.dumps(request, ensure_ascii=False, indent=2))
print("RESPONSE:", json.dumps(discovery, ensure_ascii=False, indent=2))
request, called, headers = await lab.mcp_request("tools/call", {"name": "list_topic_artifacts", "arguments": {"topic_name": "mcp"}})
print("TOOL REQUEST:", json.dumps(request, ensure_ascii=False, indent=2))
print("TOOL RESPONSE:", json.dumps(called, ensure_ascii=False, indent=2))
assert "error" not in called, called

### 7.2. Подключение клиента прежней редакции

Тот же SDK поддерживает совместимость с прежним рукопожатием. Отправим `initialize` с редакцией 2025-11-25.
Сравните ответ и HTTP-заголовки с предыдущим вызовом. Наличие совместимости на этом сервере не означает,
что любая пара старого и нового клиента будет поддерживать все возможности друг друга.

In [ ]:
old_request, old_response, old_headers = await lab.mcp_request(
    "initialize", {"protocolVersion": "2025-11-25", "capabilities": {}, "clientInfo": {"name": "mipt-demo", "version": "1.0"}},
    version="2025-11-25", request_meta=False,
)
print(json.dumps(old_response, ensure_ascii=False, indent=2))
print("Mcp-Session-Id:", old_headers.get("mcp-session-id", "не выдан"))
mcp_http.stop()

## 8. ВкусВилл и несколько MCP-серверов

Второй публичный сервер помогает подобрать продукты и создать ссылку на корзину.
Сначала посмотрим его каталог. Отдельные персональные возможности требуют авторизации пользователя.
Публичный поиск может показывать товары, доступность которых позже зависит от адреса доставки.

Для общего сценария используем конфигурацию с двумя серверами. Адаптер добавляет префиксы к именам инструментов,
чтобы приложение различало возможности разных серверов. Агент получает общий набор tools.

In [ ]:
connections = {"mcpServers": {
    "tutu": {"url": "https://mcp.tutu.ru/mcp"},
    "vkusvill": {"url": "https://mcp.vkusvill.ru/mcp"},
}}
async with MCPAdapter(connections) as adapter:
    combined_tools = await adapter.list_tools()
    print("Инструментов:", len(combined_tools))
    for t in combined_tools:
        print(t.name)

### 8.1. Один агент с инструментами двух серверов

Пользовательская задача содержит две части: поезд и перекус. В этой архитектуре один агент сам выбирает инструменты
обоих сервисов. По результату поиска продуктов он составляет аргументы создания ссылки на корзину.
Ссылка на корзину не означает оформление или оплату заказа.

Позже выполним ту же задачу с отдельным агентом по поездкам и сравним распределение работы.

In [ ]:
TRIP_AND_SNACK = (f"Еду {TRIP_DATE} из Москвы в Казань самым дешёвым прямым поездом. "
                  "Найди его и подбери во ВкусВилле перекус на одного из 3–4 продуктов. Дай ссылку на корзину. Не оформляй и не оплачивай заказ.")
model = make_model()
try:
    async with MCPAdapter(connections) as adapter:
        combined = create_deep_agent(model=model, tools=await adapter.list_tools(),
            system_prompt="Билеты ищи на Туту, продукты — во ВкусВилле. После поиска и получения ссылки дай итоговый ответ и заверши работу.")
        combined_result = await combined.ainvoke({"messages": [HumanMessage(content=TRIP_AND_SNACK)]}, config={"recursion_limit": 80})
finally:
    await model._client.aclose()
for message in combined_result["messages"]:
    message.pretty_print()

### 8.2. Доступ к личному кабинету: обзор OAuth

В персональном сценарии пользователь разрешает приложению доступ к определённым возможностям.
Схема: регистрация OAuth-клиента → переход пользователя на страницу авторизации → согласие → код авторизации → токен доступа.
PKCE связывает получение кода и его обмен. Пользователь может отозвать доступ.

В этом блокноте используем публичные операции. Полный вход в личный кабинет не выполняется.
История заказов и персональные скидки не нужны для основного примера.
Описание реализации: [статья ВкусВилла об OAuth 2.0 + PKCE](https://habr.com/ru/companies/vkusvill/articles/1094732/).

**Практика:** добавьте в `course_mcp.py` инструмент подсчёта файлов в папке учебных материалов. Проверьте его наличие
через `list_tools()`, вызовите обычным клиентом и затем агентом. Сохраните ограничение доступа папкой `data/course`.

# Продолжение: взаимодействие агентов и приложений

## 9. A2A: задача для удалённого агента

В A2A клиент обращается к агенту, который сам выполняет свою часть задачи. У агента могут быть собственные инструменты,
контекст и внутренняя логика. Agent Card описывает адрес, поддерживаемые интерфейсы и skills агента.
Skills в Agent Card являются описаниями возможностей; формат пакета `SKILL.md` рассматривается отдельно в дополнительном разделе.

Основной обмен: обнаружение карточки → `SendMessage` → задача → завершённый статус → `artifacts` с результатом.
Для продолжения диалога используются идентификаторы контекста. В следующих ячейках каждый запрос начинает новую задачу.

Сравним два способа публикации **одной и той же** фабрики `agents.travel.travel_graph`.
В ней одинаковы модель, инструкции и инструменты MCP Туту. Различается серверная реализация A2A.

In [ ]:
display(Markdown("```python\n" + (ROOT / "agents/travel.py").read_text(encoding="utf-8") + "\n```"))

## 10. Способ A: сервер на a2a-sdk

В `servers/a2a_sdk.py` явно определены Agent Card и `TravelExecutor`.
Исполнитель создаёт задачу, отмечает начало работы, вызывает общий граф, сохраняет ответ в artifact и завершает задачу.
Показ внутренней истории агента записывается в журнал сервера. Хранилище задач в этом примере находится в памяти процесса.

In [ ]:
display(Markdown("```python\n" + (ROOT / "servers/a2a_sdk.py").read_text(encoding="utf-8") + "\n```"))
sdk_server = lab.start_server("a2a-sdk", "servers/a2a_sdk.py", port=9100, ready_path="/.well-known/agent-card.json")
sdk_card = await lab.discover_agent("http://127.0.0.1:9100")
print(json.dumps(sdk_card, ensure_ascii=False, indent=2))

### 10.1. Сообщение, задача и artifact

В небольшом JSON-RPC-клиенте видны сообщения протокола. Он использует адрес интерфейса из карточки.
Вызов `SendMessage` здесь блокирующий: получаем результат после окончания работы.
Рассмотрите отправленный запрос и статус задачи, затем прочитайте artifact.

In [ ]:
sdk_reply = await lab.ask_a2a(sdk_card, QUESTION)
print("REQUEST:", json.dumps(sdk_reply.request, ensure_ascii=False, indent=2))
print("TASK:", sdk_reply.task["id"], sdk_reply.task["status"])
print("ARTIFACT:", sdk_reply.text)
print("\nИстория агента на сервере:\n", sdk_server.read_log(lines=70))

## 11. Способ B: встроенный A2A в LangGraph Agent Server

Второй способ использует ту же фабрику агента. `langgraph.json` связывает имя `travel` с экспортом `graph`.
Команда `langgraph dev` запускает локальный Agent Server; A2A endpoint и управление запуском предоставляет платформа.
Фабрика графа остаётся async context manager, поэтому MCP-подключение открыто на время выполнения.

Agent Server управляет состоянием запуска; фабрика возвращает агента без отдельного checkpointer.

In [ ]:
print((ROOT / "servers/agent_server.py").read_text(encoding="utf-8"))
print((ROOT / "langgraph.json").read_text(encoding="utf-8"))
platform_server = lab.start_server("agent-server", port=9300, ready_path="/ok", agent_server=True)
platform_card = await lab.platform_card()
print(json.dumps(platform_card, ensure_ascii=False, indent=2))

### 11.1. Тот же клиент и тот же запрос

Меняется только карточка подключения. Запрос, клиентская функция и фабрика агента остаются прежними.
Внешний поиск выполняется дважды, поэтому цена и формулировки могут отличаться.
Сравните маршрут, дату, вызов инструмента и завершённый статус задачи.

In [ ]:
platform_reply = await lab.ask_a2a(platform_card, QUESTION)
print("TASK:", platform_reply.task["id"], platform_reply.task["status"])
print("ARTIFACT:", platform_reply.text)
print("\nОтвет сервера SDK:\n", sdk_reply.text)

In [ ]:
context_id = platform_reply.task["contextId"]
async with httpx.AsyncClient(timeout=30) as http:
    state_response = await http.get(f"http://127.0.0.1:9300/threads/{context_id}/state")
    state_response.raise_for_status()
    server_messages = state_response.json()["values"]["messages"]
print("Вызовы инструментов внутри Agent Server:", [call["name"] for m in server_messages for call in m.get("tool_calls", [])])

### 11.2. Сравнение серверных реализаций

| Слой | a2a-sdk | Agent Server |
|---|---|---|
| Логика агента | Общая фабрика Deep Agents + MCP Туту | Та же фабрика |
| Agent Card и обработка запроса | Настраиваем в своём сервере | Предоставляет платформа |
| Задачи и состояние | Явный TaskUpdater и InMemoryTaskStore | Серверное управление запуском и состоянием |
| Клиент основного примера | Общий JSON-RPC | Тот же JSON-RPC |

Клиент `lab.ask_a2a` получает адрес из Agent Card и отправляет JSON-RPC-запрос.
Найдите в ответе идентификатор задачи, `status.state` и текст в `artifacts`.
Сопоставьте эти поля у двух серверов: клиент использует их для чтения результата.
Источники: [A2A endpoint](https://docs.langchain.com/langsmith/server-a2a),
[фабрики графов](https://docs.langchain.com/langsmith/graph-rebuild).

## 12. Координатор: A2A и MCP в одном приложении

Теперь в системе два агента. Координатор работает в ноутбуке, агент по поездкам — в серверном процессе.
Координатор вызывает `travel_agent` с запросом на естественном языке. Внутри функции находится A2A-клиент.
Удалённый агент сам выбирает инструменты Туту. Продукты координатор ищет через MCP ВкусВилла.

Следующая функция одинакова для обоих серверов. Выбор карточки определяет, куда уйдёт запрос о поездке.
Внешний результат не раскрывает координатору всю внутреннюю историю удалённого агента: её смотрим на сервере.

In [ ]:
async def coordinate(target_card):
    @tool
    async def travel_agent(request: str) -> str:
        """Задать удалённому агенту вопрос о поездке: направление, дата, условия поиска."""
        reply = await lab.ask_a2a(target_card, request)
        print("A2A:", reply.task["status"]["state"])
        return reply.text

    model = make_model()
    try:
        async with MCPAdapter("https://mcp.vkusvill.ru/mcp") as adapter:
            coordinator = create_deep_agent(
                model=model, tools=[travel_agent, *await adapter.list_tools()],
                system_prompt="Билеты ищи через travel_agent, продукты — через ВкусВилл. Получив билетные сведения и ссылку на корзину, дай итоговый ответ. Заказ не оформляй.",
            )
            return await coordinator.ainvoke({"messages": [HumanMessage(content=TRIP_AND_SNACK)]}, config={"recursion_limit": 80})
    finally:
        await model._client.aclose()

In [ ]:
coordinator_sdk = await coordinate(sdk_card)
for message in coordinator_sdk["messages"]:
    message.pretty_print()

In [ ]:
coordinator_platform = await coordinate(platform_card)
for message in coordinator_platform["messages"]:
    message.pretty_print()

Сравните вызовы координатора в двух запусках: обращение к удалённому агенту, поиск продуктов и создание ссылки.
Сравните также первый вариант из раздела 8, где один агент имел прямые MCP-подключения к обоим сервисам.
Обсудите, какую часть задачи удобно поручить отдельному агенту и какие дополнительные расходы даёт ещё один модельный запуск.

In [ ]:
sdk_server.stop()
platform_server.stop()

## 13. AG-UI: события для пользовательского приложения

Пользовательское приложение должно показывать состояние запуска: агент начал работу, вызвал инструмент,
получил данные, формирует ответ, завершил выполнение. AG-UI задаёт структуру таких событий.
Здесь транспорт — HTTP POST с ответом в виде SSE-потока. SSE передаёт последовательность событий в одном открытом ответе.

Запускаем FastAPI-приложение с агентом в отдельном процессе. Ноутбук играет роль клиента:
отправляет запрос и обрабатывает события по мере поступления. Полный интерфейс чата для этого опыта не нужен.
Служебная функция возвращает также список событий, чтобы после запуска изучить их поля.

In [ ]:
agui_server = lab.start_server("agui", "servers/agui.py", port=9200, ready_path="/openapi.json", args=["--port", "9200"])
events = await lab.agui_events(9200, QUESTION)
print("\nЧисло событий каждого типа:", dict(Counter(e["type"] for e in events)))

### 13.1. Что приложение может показать по событиям

| Событие | Возможное действие приложения |
|---|---|
| `RUN_STARTED` | Показать начало работы |
| `TOOL_CALL_START`, `TOOL_CALL_ARGS` | Показать операцию и её параметры |
| `TOOL_CALL_RESULT` | Показать полученные сведения |
| `TEXT_MESSAGE_CONTENT` | Дописать очередной фрагмент текста |
| `STATE_SNAPSHOT` | Обновить представление состояния |
| `RUN_FINISHED`, `RUN_ERROR` | Завершить индикатор работы или показать ошибку |

Названия событий не определяют внешний вид приложения. Клиент выбирает, как визуализировать каждый тип.
Посмотрим одно событие вызова и одно событие результата. Большой ответ инструмента выводим отдельно от общего потока.

In [ ]:
for kind in ["TOOL_CALL_START", "TOOL_CALL_RESULT", "STATE_SNAPSHOT", "RUN_FINISHED"]:
    event = next((e for e in events if e["type"] == kind), None)
    print(kind, "\n", json.dumps(event, ensure_ascii=False, indent=2)[:2000], "\n")
agui_server.stop()

## 14. A2UI: описание интерфейса в результате агента

A2UI описывает компоненты интерфейса и связанные данные. Например, агент может вернуть карточку поезда с временем
и ценой. Рендерер клиента выбирает зарегистрированные компоненты и строит из них экран.
AG-UI в этом опыте передаёт события выполнения, а инструмент генерации возвращает описание A2UI.

В закреплённом адаптере генератор использует A2UI **v0.9**. Текущую спецификацию v1.0 рассматриваем как отдельную редакцию;
совместимость рендереров разных версий автоматически не предполагается.
В следующей ячейке используем тот же сервер, включив `--a2ui`.

In [ ]:
a2ui_server = lab.start_server("a2ui", "servers/agui.py", port=9201, ready_path="/openapi.json", args=["--a2ui", "--port", "9201"])
a2ui_events = await lab.agui_events(9201, QUESTION + " Покажи один результат карточкой A2UI и заверши ответ.")
operations = lab.a2ui_operations(a2ui_events)
print("\nA2UI JSON:\n", json.dumps(operations, ensure_ascii=False, indent=2))

### 14.1. Как связаны JSON и изображение карточки

`createSurface` создаёт поверхность интерфейса. `updateComponents` задаёт компоненты с идентификаторами и связями.
Контейнер `root` содержит дочерние элементы; `Text` показывает текст, `Card` группирует сведения, `Row` располагает их рядом.
`updateDataModel` позволяет обновлять связанные данные без замены структуры компонентов.

Ниже — **учебный статический просмотр базовых компонентов v0.9** в HTML-выводе ноутбука.
Он показывает Text, Card, Column, Row и Divider. Кнопки не выполняют действия; List с шаблоном и связанные данные
требуют полноценного рендерера. Сравните фактический JSON с полученным представлением.

In [ ]:
display(HTML(lab.preview_a2ui(operations)))
a2ui_server.stop()

Инструмент генерации A2UI подключается в `servers/agui.py`; его реализация находится в `compat/a2ui_tool.py`.
Вызов агента возвращает описание компонентов, а `lab.preview_a2ui` отображает их в блокноте.
Источники: [AG-UI](https://docs.ag-ui.com/), [A2UI](https://a2ui.org/), [пример A2UI](https://a2ui.org/quickstart/).

## 15. Agent Client Protocol: редактор и агентный процесс

ACP стандартизует взаимодействие редактора с агентом-программистом: начало сессии, запросы, обновления и другие операции.
В опыте ноутбук выполняет роль минимального клиента. Агент запускается отдельным процессом и общается через stdio.
Для stdout важно сохранять только сообщения протокола; диагностический вывод сервера направлен в stderr.

Клиент создаёт сессию с рабочим каталогом, отправляет prompt и читает уведомления `session/update` до ответа о завершении.
HTTP-сервер и открытый порт здесь не нужны. Используем Popen и отдельный поток чтения:
так запуск stdio-процесса не зависит от поддержки asyncio subprocess в цикле Jupyter на Windows.

In [ ]:
acp_client = lab.ACPClient()
request, initialized, updates = acp_client.request("initialize", {
    "protocolVersion": 1, "clientCapabilities": {}, "clientInfo": {"name": "mipt-notebook", "version": "1.0"},
})
print("INITIALIZE:", json.dumps(initialized, ensure_ascii=False, indent=2))
acp_workspace = ROOT / ".runtime/acp-workspace"
acp_workspace.mkdir(parents=True, exist_ok=True)
_, session, _ = acp_client.request("session/new", {"cwd": str(acp_workspace), "mcpServers": []})
print("SESSION:", session)

In [ ]:
try:
    request, completion, updates = acp_client.request("session/prompt", {
        "sessionId": session["sessionId"], "prompt": [{"type": "text", "text": "Привет! Объясни в одном коротком предложении, что ты готов принять задачу."}],
    })
    print("REQUEST:", json.dumps(request, ensure_ascii=False, indent=2))
    for event in updates:
        print(json.dumps(event, ensure_ascii=False))
    print("COMPLETION:", completion)
    assert completion["stopReason"] == "end_turn"
finally:
    acp_client.stop()

Возможности клиента и сервера согласуются в `initialize`. Обработка файлов, подтверждение действий и другие функции
редактора требуют дополнительных методов клиента. Наш короткий пример показывает минимальный обмен текстом.
[Документация Deep Agents ACP](https://docs.langchain.com/oss/python/deepagents/acp).

## 16. Коммерция и платёжные протоколы — обзор

Агент, который нашёл товар, должен согласовать условия покупки, получить разрешение пользователя и передать
платёжные сведения соответствующим участникам. Разные протоколы покрывают разные части этого процесса.
В этом разделе выполняемых платёжных примеров нет.

| Протокол | Обсуждаемый предмет | Пример сообщения или этапа |
|---|---|---|
| **Agentic Commerce Protocol, ACP** | Взаимодействие агента и продавца, checkout | Создание и уточнение оформления покупки |
| **Universal Commerce Protocol, UCP** | Жизненный цикл коммерции | Поиск → корзина → checkout → работа с заказом |
| **Agent Payments Protocol, AP2** | Проверяемое делегирование и разрешение оплаты | CheckoutMandate и PaymentMandate |
| **x402** | Оплата доступа к HTTP-ресурсу | Ответ 402 с условиями, затем запрос с подтверждением оплаты |

Сравните два сценария: покупка набора продуктов и платный запрос к API.
В первом нужны позиции заказа, условия и разрешение на оплату. Во втором требуется связать доступ к ресурсу с оплатой запроса.
AP2 может применяться как расширение UCP для подтверждения полномочий и согласия на оплату.

Аббревиатура ACP встречается также у **Agent Client Protocol**, который мы только что использовали для общения с агентным процессом.
При обсуждении всегда уточняйте полное название.

Источники: [Agentic Commerce Protocol](https://www.agenticcommerce.dev/), [UCP](https://ucp.dev/),
[AP2](https://ap2-protocol.org/), [связь UCP и AP2](https://ucp.dev/documentation/ucp-and-ap2/), [x402](https://x402.org/).

## 17. Итог основного занятия

Мы подключили готовые серверы, опубликовали свои tools/resources/prompts, собрали вход модели из MCP-контекста,
выполнили задачу одним агентом с двумя серверами и двумя агентами с A2A. Затем посмотрели события приложения,
описание интерфейса и общение с агентным процессом по stdio.

**Контрольные вопросы**

1. Кто читает MCP-resource и решает передать его модели?
2. Что возвращает MCP-prompt и чем его получение отличается от запуска модели?
3. Как координатор узнаёт адрес A2A-сервера? Где выполняется внутренний поиск?
4. Что меняется при публикации той же фабрики через Agent Server?
5. Какое событие AG-UI сообщает об окончании запуска?
6. Кто превращает JSON A2UI в компоненты интерфейса?
7. Что получает клиент ACP из stdout дочернего процесса?

In [ ]:
lab.stop_all()
print("Локальные серверы остановлены")

# Домашнее задание: RAG и MCP

Полные условия и критерии оценки: [PDF задания](homework_mcp_knowledge_base.pdf).

Подготовьте собственный корпус документов, выполните разбиение, добавьте метаданные и постройте индекс.
Подключите **готовый MCP-сервер** поиска к агенту на LangChain, LangGraph или Deep Agents.
Агент должен вызывать поиск и возвращать top-k фрагментов с `document_id`, `chunk_id` и `source`.
Проверьте поиск на **15–20 запросах**, включая вопросы без ответа в корпусе, и приложите ручную оценку.
Требования к корпусу, состав сдачи и веса критериев приведены в PDF.
Генерация ответа со ссылками поверх найденных фрагментов — дополнительная часть по желанию.

В PDF приведены команды запуска Qdrant и пример подключения через `MCPAdapter` к Deep Agents.
Используйте указанную версию сервера и отдельное окружение: это позволяет воспроизвести конфигурацию примера.

# Дополнительный материал: Skills

## 18. Навык как пакет инструкций и файлов

Skill предоставляет агенту процедуру выполнения задачи. `SKILL.md` содержит имя, описание и инструкции;
рядом могут находиться скрипты, шаблоны и справочные материалы. По описанию агент выбирает релевантный навык,
а полный текст читает, когда приступает к задаче. Это помогает загружать подробности по мере необходимости.

Для практики используем QR-навык: из заданной строки он создаёт PNG, затем читает QR-код обратно.
Результат можно проверить независимым декодером. В папке навыка находится `SKILL.md`, а в `scripts/` — `qr_generate.py` и `qr_read.py`.
Имя `qr-code` в YAML совпадает с именем папки. Каталог `skills/` передаём агенту явно через `skills=["/skills"]`.
Источник и автор указаны в YAML-поле `metadata` файла `SKILL.md`; Python-скрипты содержат указание MIT.

Для декодирования нужна библиотека ZBar. В Windows x64 DLL входят в wheel pyzbar; при ошибке загрузки может понадобиться
Visual C++ Redistributable 2013 x64. На macOS используется Homebrew, в Debian/Ubuntu — libzbar0.
Инструкции по ОС приведены в README. Проверка зависимости выполняется до запуска модели.

In [ ]:
print((ROOT / "skills/qr-code/SKILL.md").read_text(encoding="utf-8"))
print("Файлы:", [str(p.relative_to(ROOT / "skills/qr-code")) for p in sorted((ROOT / "skills/qr-code").rglob("*")) if p.is_file()])

### 18.1. Предварительная проверка генератора и декодера

Используем включённые в репозиторий скрипты в отдельной рабочей папке. Этот короткий запуск позволяет убедиться,
что Python-пакеты и нативная библиотека установлены, прежде чем тратить запросы модели на агентный сценарий.
Код читает только созданный в этом примере файл.

In [ ]:
import subprocess
import shutil
import tempfile

qr_runtime = ROOT / ".runtime/qr"
qr_runtime.mkdir(parents=True, exist_ok=True)
qr_text = "https://example.com/mipt-protocols"
qr_image = qr_runtime / "check.png"
for command in [
    [sys.executable, str(ROOT / "skills/qr-code/scripts/qr_generate.py"), qr_text, str(qr_image)],
    [sys.executable, str(ROOT / "skills/qr-code/scripts/qr_read.py"), str(qr_image)],
]:
    checked = subprocess.run(command, check=True, capture_output=True, text=True, encoding="utf-8")
    print(checked.stdout)
assert checked.stdout.strip() == qr_text

### 18.2. Deep Agent выполняет навык

Временная рабочая папка содержит только выбранный навык. `LocalShellBackend` даёт агенту файловые инструменты и исполнение команд.
`virtual_mode` отображает пути файловых инструментов относительно этой папки. В shell-командах используем обычные относительные пути.
Сам shell работает с правами Python-процесса; для неизвестных навыков требуется отдельная изолированная среда.
В данном примере используем включённые и просмотренные скрипты QR-навыка.

Агент должен прочитать SKILL.md, выполнить генератор и декодер, сопоставить строки и завершить ответ.
После него проверим результат ещё раз обычным Python-вызовом.

In [ ]:
from deepagents.backends import LocalShellBackend
from langgraph.checkpoint.memory import MemorySaver

skill_workspace = ROOT / ".runtime/skill-workspace"
(skill_workspace / "skills").mkdir(parents=True, exist_ok=True)
copied_skill = skill_workspace / "skills/qr-code"
if copied_skill.exists():
    shutil.rmtree(copied_skill)
shutil.copytree(ROOT / "skills/qr-code", copied_skill)
# Предыдущий результат удаляется, чтобы новая проверка не приняла его за свежий.
(skill_workspace / "result.png").unlink(missing_ok=True)
model = make_model()
try:
    skill_agent = create_deep_agent(
        model=model, backend=LocalShellBackend(root_dir=skill_workspace, virtual_mode=True,
            env={"PATH": str(Path(sys.executable).parent) + os.pathsep + os.defpath}),
        skills=["/skills"], checkpointer=MemorySaver(),
        system_prompt=("Выбери подходящий навык и следуй его инструкции. "
            f"Интерпретатор Python: {sys.executable}. "
            "Сохрани result.png и сравни декодированную строку с исходной. Все зависимости уже установлены. "
            "После успешной проверки заверши ответ."),
    )
    skill_result = await skill_agent.ainvoke({"messages": [HumanMessage(content=f"Создай QR для {qr_text} и прочитай его обратно.")]},
        config={"configurable": {"thread_id": str(__import__("uuid").uuid4())}, "recursion_limit": 50})
finally:
    await model._client.aclose()
for message in skill_result["messages"]:
    message.pretty_print()

In [ ]:
verified = subprocess.run([sys.executable, str(ROOT / "skills/qr-code/scripts/qr_read.py"), str(skill_workspace / "result.png")],
                          check=True, capture_output=True, text=True, encoding="utf-8")
assert verified.stdout.strip() == qr_text
from PIL import Image
with Image.open(skill_workspace / "result.png") as image:
    display(image.copy())
print("Независимая проверка пройдена:", verified.stdout.strip())

### 18.3. Как загрузить навык обычными инструментами LangChain

Та же идея может быть реализована через два явных инструмента: список доступных навыков и загрузку выбранного SKILL.md.
Этот небольшой пример показывает этап выбора инструкций; исполнение QR-скриптов уже продемонстрировано выше.
Полный текст каждого навыка не обязательно помещать в system prompt при создании агента.

In [ ]:
@tool
def list_skills() -> list[dict]:
    """Перечислить доступные локальные навыки."""
    return [{"name": "qr-code", "description": "Генерация и чтение QR-кодов"}]

@tool
def load_skill(name: str) -> str:
    """Загрузить инструкции выбранного навыка."""
    if name != "qr-code":
        raise ValueError("Доступен навык qr-code")
    return (ROOT / "skills/qr-code/SKILL.md").read_text(encoding="utf-8")

from langchain.agents import create_agent
model = make_model()
try:
    skill_loader = create_agent(model=model, tools=[list_skills, load_skill],
        system_prompt="Узнай, какие навыки доступны, загрузи подходящий и объясни, какие шаги потребуются для QR-кода. Скрипты сейчас не выполняй.")
    loaded = await skill_loader.ainvoke({"messages": [HumanMessage(content="Как создать QR-код и проверить его чтение?")]},
                                     config={"recursion_limit": 20})
finally:
    await model._client.aclose()
for message in loaded["messages"]:
    message.pretty_print()

## Словарь и источники

- **Host** — приложение, которое использует MCP-клиента и модель.
- **Tool** — операция с описанием аргументов и результата; **resource** — адресуемые данные; **prompt** — параметризованный шаблон сообщений.
- **Agent Card** — описание интерфейсов и возможностей A2A-агента; **task** — задача; **artifact** — результат задачи.
- **SSE** — Server-Sent Events, поток событий от сервера по HTTP.
- **Renderer** — клиентская часть, строящая интерфейс из описания компонентов.
- **Skill** — пакет инструкций и вспомогательных материалов для выполнения задачи.

Основные первоисточники:
[MCP](https://modelcontextprotocol.io/specification/2026-07-28),
[MCP Python SDK](https://py.sdk.modelcontextprotocol.io/),
[LangChain MCP](https://docs.langchain.com/oss/python/langchain/mcp),
[A2A](https://a2a-protocol.org/latest/specification/),
[AG-UI](https://docs.ag-ui.com/), [A2UI](https://a2ui.org/),
[Agent Client Protocol](https://agentclientprotocol.com/overview/introduction),
[Skills Deep Agents](https://docs.langchain.com/oss/python/deepagents/skills).

In [ ]:
lab.stop_all()
print("Работа завершена; локальные серверы остановлены")